# SlowBurn Demo

**Cost-Sustainable Concurrent Execution for Long-Horizon LLM Agents**

This notebook demonstrates SlowBurn end-to-end:

1. **Quick start** — two-line LLM setup with `create_llm()`
2. **Vision support** — send images to the LLM
3. **Global config** — `temp_config()` for environment-specific defaults
4. **Research agent** — a real ReAct agent with web search and file writing, running under a dollar budget
5. **Cost report** — Markdown and LaTeX tables from the agent run

## Architecture

![SlowBurn Architecture](../figures/architecture.png)

In [ ]:
import os, sys, time
from pathlib import Path
from dotenv import load_dotenv

# Load API keys
load_dotenv(Path("../.env"))

if os.getenv("OPENROUTER_API_KEY"):
    API_KEY = os.getenv("OPENROUTER_API_KEY")
    FAST_MODEL = "openrouter/google/gemini-2.0-flash-001"
    AGENT_MODEL = "openrouter/z-ai/glm-4.5-air"  # cheap + good at tool calling
else:
    API_KEY = os.getenv("OPENAI_API_KEY", "")
    FAST_MODEL = "gpt-4o-mini"
    AGENT_MODEL = "gpt-4o-mini"

# Add src/ to path so imports work from the notebook
sys.path.insert(0, str(Path("../src").resolve()))

print(f"Fast model:  {FAST_MODEL}")
print(f"Agent model: {AGENT_MODEL}")
print(f"API key:     {'loaded' if API_KEY else 'MISSING'}")

---
## 1. Quick Start: Two-Line LLM Setup

In [ ]:
from slowburn import create_llm

llm = create_llm(
    model=FAST_MODEL,
    api_key=API_KEY,
    budget_usd=1.00,
    window="hourly",
    max_tokens=200,
    temperature=0.3,
)

result = llm.call_llm(
    prompt="Explain what backpressure means in distributed systems, in two sentences.",
).result(timeout=30.0)

print(result)

reporter = llm.get_reporter().result(timeout=5.0)
print(f"\nCost: ${reporter.total_cost():.6f}")

### Batch calls (concurrent)

All three prompts execute concurrently on the asyncio event loop.

In [ ]:
start = time.time()
results = llm.call_llm_batch(
    prompts=[
        "Name one advantage of rate limiting in APIs.",
        "Name one advantage of circuit breakers in microservices.",
        "Name one advantage of exponential backoff in retries.",
    ]
).result(timeout=30.0)
elapsed = time.time() - start

for r in results:
    print(f"  - {r.strip()}")
print(f"\n3 concurrent calls in {elapsed:.1f}s")

---
## 2. Vision: Send Images to the LLM

In [ ]:
image_dir = Path("../tests/fixtures/images")
test_images = sorted(image_dir.glob("test_image_*.jpg"))
print(f"{len(test_images)} test images available")

desc = llm.call_llm(
    prompt="Describe this image in one detailed sentence.",
    images=[test_images[6]],  # coffee mug on red background
).result(timeout=30.0)

print(f"\n{test_images[6].name}: {desc}")

In [ ]:
llm.stop()  # done with quick-start worker

---
## 3. Global Config: `temp_config()` for Environment-Specific Defaults

In [ ]:
from slowburn import slowburn_config, temp_config

cfg = slowburn_config.defaults
print("Current defaults:")
print(f"  temperature:       {cfg.temperature}")
print(f"  max_tokens:        {cfg.max_tokens}")
print(f"  budget_usd:        ${cfg.budget_usd}")
print(f"  chars_per_token:   {cfg.chars_per_token}")
print(f"  safety_multiplier: {cfg.token_safety_multiplier}")

print()

# temp_config scopes overrides — everything reverts on exit
with temp_config(temperature=0.0, max_tokens=50, budget_usd=0.10):
    print(f"Inside temp_config:  temperature={slowburn_config.defaults.temperature}, "
          f"max_tokens={slowburn_config.defaults.max_tokens}, "
          f"budget_usd=${slowburn_config.defaults.budget_usd}")

print(f"After temp_config:   temperature={cfg.temperature}, "
      f"max_tokens={cfg.max_tokens}, "
      f"budget_usd=${cfg.budget_usd}  (restored)")

---
## 4. Research Agent Under Dollar Budget

This runs the **actual `demo_native_research_agent`** — a ReAct agent that:
- Searches the web with DuckDuckGo
- Takes notes by writing files to a sandboxed workspace
- Synthesizes a research report

Every LLM call is cost-tracked. When the budget is exhausted, SlowBurn blocks (backpressure) rather than crashing.

In [ ]:
import asyncio
from datetime import datetime
from concurry import CallLimit, LimitSet, RateLimit
from slowburn.limits import CostLimit
from slowburn.reporter import CostReporter

# Import the shared agent loop and tools from the demos/lib/ folder
sys.path.insert(0, str(Path(".").resolve()))
from lib.agent_loop import run_agent
from lib.tools import TOOL_SCHEMAS, execute_tool_call

BUDGET_USD = 0.15
MAX_TOKENS = 600
MAX_STEPS = 12

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
runs_dir = Path(f"../runs/demo_notebook/{timestamp}")
runs_dir.mkdir(parents=True, exist_ok=True)

limit_set = LimitSet(
    limits=[
        CostLimit(budget_usd=BUDGET_USD, window_seconds=3600),
        RateLimit(key="input_tokens", window_seconds=60, capacity=500_000),
        RateLimit(key="output_tokens", window_seconds=60, capacity=100_000),
        CallLimit(window_seconds=60, capacity=100),
    ],
    mode="thread",
    shared=True,
)
reporter = CostReporter()

def tool_executor(name, args):
    return execute_tool_call(name, args, workspace=runs_dir)

print(f"Agent model: {AGENT_MODEL}")
print(f"Budget:      ${BUDGET_USD}")
print(f"Max steps:   {MAX_STEPS}")
print(f"Workspace:   {runs_dir}")

In [ ]:
RESEARCH_TASK = (
    "Research the cost of running LLM agents in production. "
    "Search the web for real data on API costs for GPT-4, Claude, and Gemini. "
    "Find specific dollar amounts from benchmarks like SWE-bench and Tau-bench. "
    "Write your findings to a file called 'cost_analysis.md' with sources."
)

SYSTEM_PROMPT = """\
You are a thorough research agent. You have access to these tools:
- search_web: Search the web with DuckDuckGo to find real information
- write_file: Save your research notes and reports to files
- read_file: Read files you've previously written
- list_dir: See what files exist in your workspace

For the research task:
1. Use search_web 2-3 times with different queries to gather information
2. Use write_file to save a structured report with real sources and URLs
3. Be specific: cite actual numbers, paper names, and URLs from search results

IMPORTANT: You MUST use search_web to find real data. Do NOT make up facts."""

print(f"Task: {RESEARCH_TASK[:80]}...")
print("\nRunning agent...\n")

start = time.time()
result = asyncio.run(run_agent(
    model=AGENT_MODEL,
    task=RESEARCH_TASK,
    tools=TOOL_SCHEMAS,
    tool_executor=tool_executor,
    limit_set=limit_set,
    reporter=reporter,
    api_key=API_KEY,
    system_prompt=SYSTEM_PROMPT,
    max_steps=MAX_STEPS,
    max_tokens=MAX_TOKENS,
    temperature=0.3,
    verbose=True,
    log_dir=runs_dir / "task_01",
))
elapsed = time.time() - start

print(f"\nAgent finished in {elapsed:.1f}s")
print(f"Steps: {result['steps']}, Tool calls: {result['tool_calls']}")
print(f"Cost: ${result['cost_usd']:.6f}")

### Agent's output

In [ ]:
# Show the research report the agent wrote
report_path = runs_dir / "cost_analysis.md"
if report_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))
else:
    print("Agent's final answer:")
    print(result["result"][:500])

---
## 5. Cost Report

The `CostReporter` accumulated every LLM call from the agent run.

In [ ]:
from IPython.display import Markdown, display

print(f"Total LLM calls: {reporter.num_calls}")
print(f"Total cost:      ${reporter.total_cost():.6f}")
print(f"Budget used:     {reporter.total_cost() / BUDGET_USD * 100:.1f}% of ${BUDGET_USD}")
print()

# Markdown table
display(Markdown(reporter.to_markdown()))

In [ ]:
# LaTeX table (for papers)
print(reporter.to_latex())

In [ ]:
# Files the agent created
print(f"Files in workspace ({runs_dir}):")
for f in sorted(runs_dir.rglob("*")):
    if f.is_file() and not str(f.name).startswith("."):
        print(f"  {f.relative_to(runs_dir)}: {f.stat().st_size:,} bytes")

# Save cost report to JSON
json_path = reporter.to_json(path=runs_dir / "_cost_report.json")
print(f"\nCost report saved to: {json_path}")

---

**That's SlowBurn.** A real research agent ran under a $0.15 budget,
searched the web, wrote files, and every LLM call was cost-tracked.
If the budget had been exhausted, the agent would have slowed down (backpressure)
rather than crashing.